# Numerical analysis with NumPy
### Session 11 · NumPy & Numerical Computing

**Question:** How can we describe, calculate across, and visualise a whole collection of numerical measurements without losing the meaning of its rows and columns?

You will analyse **synthetic** dispatch readings for a retailer's East, Central, and West hubs over 14 consecutive days. Each value is a number of **units dispatched by one hub on one day**. You will turn 42 readings into a day-by-hub array, compare them with nominal daily benchmarks, summarise the data, and make three charts.

**How to work:** Run cells in order from a fresh kernel. Read each prediction prompt *before* running the next cell. Supplied example cells run immediately. In numbered **TODO** cells, replace each `...` with your own expression, then rerun that cell and continue. An unfinished cell deliberately stops with a message naming its task. If you change an earlier answer, use **Restart & Run All** after completing the remaining TODOs.

You need Python 3, NumPy, Matplotlib, and a Jupyter notebook environment. If an import reports `ModuleNotFoundError`, ask your trainer which Python environment to use or install the missing package in your own environment, then restart the kernel. No data download, account, earlier project, or image file is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)

## 1. From two orders to aligned numerical values

In an earlier retailer example, two orders each had a unit price of 250.0, a 10% discount, and 5% tax **after** the discount. Their quantities were 4 and 5. The numeric inputs can be aligned by position: position 0 is the first order and position 1 is the second. Business records remain useful for describing orders; a numeric array is useful when the same arithmetic applies to a collection of numbers.

**Predict:** What does Python list `+` do? What does NumPy array `+` do? Then run this short bridge.

In [ ]:
print("List +:", [1, 2] + [3, 4])
print("Array +:", np.array([1, 2]) + np.array([3, 4]))

quantities = np.array([4, 5])
unit_prices = np.array([250.0, 250.0])
discount_rate = 0.10
tax_rate = 0.05
order_totals = quantities * unit_prices * (1 - discount_rate) * (1 + tax_rate)
print("Discount, then tax:", order_totals)

The two totals are **945.00** and **1181.25**. NumPy performs the arithmetic on matching positions without writing a separate expression for each order. That only makes sense because the positions have the same order in both arrays.

## 2. Arrays carry structure as well as values

An array's **shape** gives its length along each axis; `ndim` counts axes; `size` counts values; `dtype` describes the element type. A 2-D array is often called a **matrix** and a 1-D array a **vector**. Those names do not determine what a row or column means: we must supply the labels.

**Predict:** For the tiny array below, which value is at row index 1, column index 2? How many values and axes does it have? Index 1 is the *second* row.

In [ ]:
tiny = np.array([[8, 9, 10], [11, 12, 13]])
print(tiny)
print("shape:", tiny.shape, "ndim:", tiny.ndim, "size:", tiny.size, "dtype:", tiny.dtype)
print("row 1, column 2:", tiny[1, 2])
print("counting sequence:", np.arange(1, 5))
print("zeros:", np.zeros((2, 3)))
print("ones:", np.ones(3))
print("evenly spaced values:", np.linspace(0, 1, 3))
print("mixed numeric dtype:", np.array([1, 2.5]).dtype)

NumPy normally uses one dtype for all values in an array. Mixing whole and decimal numbers can lead to a floating dtype. The exact default integer width can differ by environment; inspect `dtype` rather than assuming one.

### Select, slice, and filter on a small grid

`grid[row, column]` selects one value. A colon means “take all positions” on that axis. A comparison produces a **Boolean mask** with the same shape as the grid; using the full mask to select values produces a **1-D result**. Predict the shapes before running.

In [ ]:
grid = np.array([[2, 5, 8], [3, 6, 9]])
mask = grid > 5
print("one row:", grid[1, :], "shape:", grid[1, :].shape)
print("one column:", grid[:, 2], "shape:", grid[:, 2].shape)
print("mask:", mask, "shape:", mask.shape)
print("selected values:", grid[mask], "shape:", grid[mask].shape)

Basic slicing commonly returns a **view**: changing the slice can change its source. Boolean selection returns a copy. The example uses a separate array so our dispatch input stays untouched. Use `.copy()` when you need an independent result from a basic slice.

In [ ]:
sample = np.array([2, 5, 8])
slice_view = sample[:2]
filtered_copy = sample[sample > 4]
slice_view[0] = 20
filtered_copy[0] = 50
print("source after edits:", sample)
print("filtered copy after edit:", filtered_copy)

### One operation, many values

Scalar arithmetic applies to each element. Equally shaped arrays combine position by position. **Broadcasting** also lets compatible smaller shapes align from the *right*: `(14, 3)` and `(3,)` fit because their final axis lengths are both 3. A length-14 vector does **not** directly align with the final axis of length 3. Compatible shapes alone are not enough: each position must also refer to the correct hub.

In [ ]:
example = np.array([[10, 20, 30], [11, 21, 31]])
print("scalar arithmetic:", example + 2)
print("same-shape arithmetic:", example + np.ones((2, 3)))
print("one number per column:", example / np.array([10, 20, 30]))

## 3. The 14-day dispatch review

The next cell is **supplied input**. `raw_dispatch_units` is flat and in **day-major order**: East, Central, West for Day 1, then East, Central, West for Day 2, and so on. `daily_capacity` contains **nominal planned daily benchmarks**, one for each hub in the same column order. It is a comparison reference, not a physical maximum.

The 42 readings are new synthetic fulfilment data; they do not come from the earlier sales orders. Run this cell as given. No transcription or file loading is needed.

In [ ]:
hub_names = ["East Hub", "Central Hub", "West Hub"]
days = np.arange(1, 15)  # human Day 1 through Day 14
daily_capacity = np.array([110, 95, 120])  # East, Central, West; units per hub per day

raw_dispatch_units = [
    102, 89, 117,   # Day 1
    110, 94, 121,   # Day 2
    115, 93, 118,   # Day 3
    107, 100, 119,  # Day 4
    119, 98, 125,   # Day 5
    92, 84, 109,    # Day 6
    87, 80, 104,    # Day 7
    105, 90, 116,   # Day 8
    112, 97, 119,   # Day 9
    108, 95, 120,   # Day 10
    116, 99, 126,   # Day 11
    121, 103, 131,  # Day 12
    96, 85, 112,    # Day 13
    91, 83, 106,    # Day 14
]

print("readings:", len(raw_dispatch_units))
print("days:", days.shape, "capacity:", daily_capacity.shape)
print("hub order:", hub_names)

### TODO 1 — Make the day-by-hub matrix

**Why:** A flat list has values but no explicit day/hub axes. Convert it to a NumPy array, then reshape to **14 rows × 3 columns**. Axis 0 must mean **day** and axis 1 **hub**. The first row should still be Day 1 in East, Central, West order.

**Predict:** `dispatch.shape`, `dispatch.ndim`, `dispatch.size`, and the shape of `dispatch[0, :]`. Replace `...` below. The supplied checks stop a wrong shape or wrong first row before later cells use it.

In [ ]:
# TODO 1: convert the flat sequence, then reshape in its existing day-major order.
dispatch = ...

if dispatch is Ellipsis:
    raise NotImplementedError("TODO 1: assign a 14-by-3 NumPy array to dispatch, then rerun this cell.")
assert isinstance(dispatch, np.ndarray) and dispatch.shape == (14, 3), "Check the 14 day rows and 3 hub columns."
assert np.array_equal(dispatch[0, :], [102, 89, 117]), "Check Day 1 and the East/Central/West column order."
print("shape:", dispatch.shape, "ndim:", dispatch.ndim, "dtype:", dispatch.dtype, "size:", dispatch.size)
print("Day 1:", dispatch[0, :])

### TODO 2 — Select days, a hub, and a condition

**Why:** A row answers “what happened on one day?”, while a column answers “what happened at one hub across days?”. Use zero-based indices: Day 2 is row index **1**; West is column index **2**.

Select Day 2, all West readings, and Days 1–7. Then compare every cell with the **fixed value 120 units**. This fixed-value mask is only a filtering exercise; it is *different* from the per-hub capacity comparison in TODO 3. Use the full mask to collect matching values.

**Predict:** shapes of the day row, hub column, week block, full mask, and filtered result. Why does the filtered result no longer show day-by-hub positions?

In [ ]:
# TODO 2: complete the selections and Boolean filtering.
day2 = ...
west_series = ...
week1_slice = ...
large_reading_mask = ...       # compare all dispatch values with 120
large_readings = ...          # use the full mask to select values

if any(value is Ellipsis for value in (day2, west_series, week1_slice, large_reading_mask, large_readings)):
    raise NotImplementedError("TODO 2: complete all five selections/filter expressions, then rerun this cell.")
assert np.shape(day2) == (3,), "Day 2 should have one value per hub."
assert np.shape(west_series) == (14,), "West should have one value per day."
assert np.shape(week1_slice) == (7, 3), "Week 1 should contain seven day rows and three hub columns."
assert np.shape(large_reading_mask) == (14, 3), "Compare every cell to form a full-size mask."
assert np.ndim(large_readings) == 1, "A full-array Boolean selection should be 1-D."
print("Day 2:", day2, "West series:", west_series)
print("Week 1 shape:", week1_slice.shape)
print("mask shape:", large_reading_mask.shape, "selected shape:", np.shape(large_readings))
print("selected values:", large_readings)

### TODO 3 — Broadcast a benchmark across all days

**Why:** Every row has the same three hub columns, so one three-value benchmark vector can be used against all 14 rows. Calculate the **dimensionless ratio** `actual / nominal benchmark`. A ratio of `1.0` means exactly at benchmark; only a ratio **greater than** `1.0` means strictly above it.

Create `exceeded` using a strict `>` comparison. Also identify exact equalities in `at_capacity`. Check Day 2: East equals its benchmark; ask whether West is above its own benchmark. A day above a *nominal* benchmark does not prove failure or an impossible shipment.

**Prediction:** Why will a `(14,)` vector not directly broadcast across a `(14, 3)` matrix? Compare shapes from the **trailing axis**.

In [ ]:
# TODO 3: use the hub-aligned capacity vector; do not loop over days.
utilisation = ...
exceeded = ...
at_capacity = ...

if any(value is Ellipsis for value in (utilisation, exceeded, at_capacity)):
    raise NotImplementedError("TODO 3: calculate utilisation, strict exceedance, and equality masks, then rerun.")
assert np.shape(utilisation) == (14, 3), "The ratio needs one value per hub-day cell."
assert np.shape(exceeded) == (14, 3) and np.asarray(exceeded).dtype == bool, "exceeded needs a full Boolean mask."
assert np.shape(at_capacity) == (14, 3) and np.asarray(at_capacity).dtype == bool, "at_capacity needs a full Boolean mask."
assert np.isclose(utilisation[1, 0], 1.0), "Day 2 East should equal its nominal benchmark."
assert not exceeded[1, 0] and at_capacity[1, 0], "Equality is not a strict exceedance."
print("ratio shape:", np.shape(utilisation), "ratio dtype:", np.asarray(utilisation).dtype)
print("Day 2 ratios (East, Central, West):", utilisation[1, :])
print("Day 2 strictly above benchmark?:", exceeded[1, :])

### TODO 4 — Reduce an axis, keep its meaning

**Why:** `axis=0` collapses the **day rows**, leaving one 14-day total for each hub. `axis=1` collapses the **hub columns**, leaving one network total for each day. Give every result a unit and label.

Compute hub totals, day totals, a mean network day, the smallest and largest network day totals, and the zero-based index of the busiest day. Convert that index to a human day using `days`. Count above-benchmark cells overall and by hub; count distinct days with at least one above-benchmark hub. Boolean `True` values count as 1 when summed. A standard deviation gives a brief measure of how much daily totals vary around their mean; it does not explain *why* they vary.

In [ ]:
# TODO 4: complete the reductions. Check each output's meaning before printing.
hub_totals = ...
day_totals = ...
mean_network_day = ...
min_daily_total = ...
max_daily_total = ...
peak_index = ...
peak_day = ...
above_by_hub = ...
above_cells = ...
above_days = ...
daily_total_std = ...

task4_values = (hub_totals, day_totals, mean_network_day, min_daily_total,
                max_daily_total, peak_index, peak_day, above_by_hub,
                above_cells, above_days, daily_total_std)
if any(value is Ellipsis for value in task4_values):
    raise NotImplementedError("TODO 4: complete every labelled summary expression, then rerun.")
assert np.shape(hub_totals) == (3,), "Hub totals should leave the hub axis."
assert np.shape(day_totals) == (14,), "Daily totals should leave the day axis."
assert np.shape(above_by_hub) == (3,), "Counts by hub should leave the hub axis."
assert 0 <= peak_index < 14 and peak_day == days[peak_index], "Translate the zero-based peak index using days."
print("14-day hub totals (units):", dict(zip(hub_names, hub_totals)))
print("daily network totals (units):", day_totals)
print("mean network day (units):", mean_network_day)
print("daily minimum / maximum (units):", min_daily_total, max_daily_total)
print("busiest human day:", peak_day, "at zero-based index:", peak_index)
print("strictly above benchmark by hub:", dict(zip(hub_names, above_by_hub)))
print("above-benchmark hub-day cells / distinct days:", above_cells, above_days)
print("standard deviation of daily network totals (units):", daily_total_std)

### TODO 5 — Split the timeline and rebuild it

**Why:** The first seven rows are Days 1–7, and the next seven are Days 8–14. Split **along the day axis**, compare the weekly totals, then stack Week 1 above Week 2. Matching shapes are not proof of matching chronology: check the actual values with `np.array_equal`.

In [ ]:
# TODO 5: split into two weeks along the day axis.
weeks = ...
if weeks is Ellipsis:
    raise NotImplementedError("TODO 5: split dispatch into two seven-day blocks, then rerun.")
assert len(weeks) == 2 and all(np.shape(week) == (7, 3) for week in weeks), "Each week needs shape (7, 3)."
week1, week2 = weeks

week1_total = ...
week2_total = ...
rebuilt = ...                 # stack Week 1 above Week 2
same_as_original = ...        # compare every value and its position
if any(value is Ellipsis for value in (week1_total, week2_total, rebuilt, same_as_original)):
    raise NotImplementedError("TODO 5: total both weeks, re-stack in time order, and compare with dispatch.")
assert np.shape(rebuilt) == (14, 3), "Rebuilt data should have the original shape."
assert same_as_original, "Recheck the order of the weekly blocks with np.array_equal."
print("Week 1 / Week 2 (units):", week1_total, week2_total)
print("Week 2 minus Week 1 (units):", week2_total - week1_total)
print("original recovered?:", same_as_original)

## 4. Turn the results into three different views

Each chart answers a different question. The figure and label code is supplied so you can focus on choosing the **right array for each axis**. Complete the two data expressions in each chart cell. Keep the labels, units and titles; run the charts only after TODOs 1–5.

### TODO 6a — Line: daily network dispatch

X should be human Days 1–14. Y should contain **one total across all hubs for each day**, not 42 individual hub readings. A line helps reveal change over time, without explaining its cause.

In [ ]:
# TODO 6a: choose the day labels and the 14 network totals you computed.
line_x = ...
line_y = ...
if line_x is Ellipsis or line_y is Ellipsis:
    raise NotImplementedError("TODO 6a: set the line chart's day and network-total arrays.")
assert np.shape(line_x) == (14,) and np.shape(line_y) == (14,), "The line needs 14 paired day/total values."
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(line_x, line_y, marker="o", color="#166c84")
ax.set(title="Daily network dispatch", xlabel="Day", ylabel="Units dispatched (all hubs)")
ax.set_xticks(days)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### TODO 6b — Bar: cumulative units by hub

X should preserve **East, Central, West** order. Y should be the three **14-day hub totals**, not the three nominal daily benchmarks. A bar chart compares category totals.

In [ ]:
# TODO 6b: choose the hub labels and three cumulative totals.
bar_x = ...
bar_y = ...
if bar_x is Ellipsis or bar_y is Ellipsis:
    raise NotImplementedError("TODO 6b: set the bar chart's hub labels and 14-day totals.")
assert len(bar_x) == 3 and np.shape(bar_y) == (3,), "The bar chart needs three matched hubs and totals."
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bar_x, bar_y, color=["#166c84", "#e2a04a", "#6b7891"])
ax.set(title="Dispatch by hub across 14 days", xlabel="Hub", ylabel="Units dispatched (14-day total)")
fig.tight_layout()
plt.show()

### TODO 6c — Scatter: paired East and West days

Each point must pair **East and West from the same day**. X is East's 14 daily readings; Y is West's 14 daily readings. A visible pattern does not by itself establish a cause or say whether either hub met its benchmark.

In [ ]:
# TODO 6c: select the East and West columns as matched daily series.
scatter_x = ...
scatter_y = ...
if scatter_x is Ellipsis or scatter_y is Ellipsis:
    raise NotImplementedError("TODO 6c: set the East and West daily series.")
assert np.shape(scatter_x) == (14,) and np.shape(scatter_y) == (14,), "The scatter needs 14 paired hub-day readings."
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(scatter_x, scatter_y, color="#a85142", s=45)
ax.set(title="East versus West: paired daily dispatch", xlabel="East Hub (units dispatched per day)",
       ylabel="West Hub (units dispatched per day)")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Interpret your evidence

Write **at least two or three findings** supported by your computed values or charts and **one limitation**. Name a day, hub, unit or comparison when relevant. You may edit the space below after running your calculations.

1. **Finding about the timeline:** _Write here._
2. **Finding about the hubs or weeks:** _Write here._
3. **Finding about benchmark comparisons or paired daily readings:** _Write here._
4. **One limitation of these data:** _Write here._

Check your wording: these are **synthetic dispatch counts**. A count above a *nominal* benchmark is not, on its own, a service failure. A scatter pattern cannot establish causation.

### If you finish early

Compare `dispatch >= daily_capacity` with your strict `exceeded` mask. Which exactly-at-benchmark cells enter the new mask? Why must the new count **not** be labelled “strictly above benchmark”?

### Transfer to the next example

A vector has one axis and this dispatch matrix has two. An RGB photograph can be represented as a three-axis numerical array: **row, column, colour channel**. The trainer will demonstrate this separately. You do not need an image file or image-processing code to finish this notebook.